# MS_C3 — Multi-Station GNN-LSTM + Stacking (Perfect Forecast Full (HYSPLIT + MET at t+h))

Trains GNN-LSTM + stacking ensemble for every station in `STATIONS_TO_RUN`.
Uses `weather_mode="perfect_forecast_full"` throughout (GNN node features and tabular
stacking base models both receive MET_COLS + HYSPLIT shifted to t+24).

**Checkpoint:** skips a station if `outputs/{station}/results/C3_metrics.csv` already exists.
A kernel restart resumes from the last incomplete station.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe
import src.models.gnn_stacking as gs

from src.config import (
    ALL_STATIONS, NEIGHBOR_STATIONS, HORIZONS, RANDOM_SEED,
    STATION_COORDS, get_station_paths,
)
from src.feature_engineering import build_feature_matrix, WEATHER_MODE_PERFECT_FULL
from src.models.gnn_stacking import (
    build_adjacency_matrix, build_gnn_sequence_dataset,
    train_gnn_lstm, predict_gnn_lstm,
    train_stacking_ensemble, predict_stacking,
)
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = WEATHER_MODE_PERFECT_FULL
# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS

print(f'Stations: {STATIONS_TO_RUN}')
print(f'Weather mode: {WEATHER_MODE}')

Stations: ['MzWarChrosci', 'MzOtwoBrzozo', 'MzWarWokalna', 'MzWarAlNiepo', 'MzLegZegrzyn', 'MzPiasPulask', 'MzWarBajkowa']
Weather mode: perfect_forecast_full


In [2]:
# Stacking base model factories (same as B3 single-station notebook)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

def make_base_models():
    return {
        'xgb': lambda: TransformedTargetRegressor(
            regressor=XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                   random_state=RANDOM_SEED, verbosity=0, tree_method='hist'),
            func=np.log1p, inverse_func=np.expm1),
        'hgb': lambda: TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(
                max_iter=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_SEED),
            func=np.log1p, inverse_func=np.expm1),
        'lgb': lambda: TransformedTargetRegressor(
            regressor=lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                        random_state=RANDOM_SEED, verbose=-1),
            func=np.log1p, inverse_func=np.expm1),
        'cat': lambda: CatBoostRegressor(iterations=300, learning_rate=0.05, depth=4,
                                         random_seed=RANDOM_SEED, verbose=0, loss_function='MAE'),
    }

In [3]:
SEQ_LEN_GNN = 24
META_FEAT_COLS_BASE = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'blh', 'pm25_now']

wall_start = time.time()
n = len(STATIONS_TO_RUN)

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'C3_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — starting ...')
    t_station = time.time()

    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # Monkeypatch TARGET so all src functions operate on the current station
    cfg.TARGET = station
    dl.TARGET = station
    fe.TARGET = station
    gs.TARGET = station
    # ALL_STATIONS_ORDERED controls node order in GNN (target node must be index 0)
    gs.ALL_STATIONS_ORDERED = [station] + [s for s in NEIGHBOR_STATIONS if s != station]

    df = dl.load_data()
    train_df, test_df = dl.train_test_split(df)

    # ── Build GNN sequence datasets ───────────────────────────────────
    X_gnn_train, y_gnn_train, A_static = build_gnn_sequence_dataset(
        train_df, seq_len=SEQ_LEN_GNN, horizons=HORIZONS, weather_mode=WEATHER_MODE
    )
    X_gnn_test, y_gnn_test, _ = build_gnn_sequence_dataset(
        test_df, seq_len=SEQ_LEN_GNN, horizons=HORIZONS, weather_mode=WEATHER_MODE
    )

    print(f'  GNN Train: {X_gnn_train.shape} | Test: {X_gnn_test.shape}')

    # 10% validation split from end of training sequences
    val_size = int(0.1 * len(X_gnn_train))
    X_gnn_tr  = X_gnn_train[:-val_size]
    y_gnn_tr  = y_gnn_train[:-val_size]
    X_gnn_val = X_gnn_train[-val_size:]
    y_gnn_val = y_gnn_train[-val_size:]

    # ── Train GNN-LSTM ────────────────────────────────────────────────
    gnn_model_path = paths['models'] / 'gnn_lstm_pfx_model.pt'
    gnn_model, _ = train_gnn_lstm(
        X_gnn_tr, y_gnn_tr, X_gnn_val, y_gnn_val, A_static,
        epochs=80, batch_size=64, patience=15, lr=1e-3,
        save_path=gnn_model_path
    )

    # GNN predictions (train for OOF alignment, test for final eval)
    gnn_train_preds = predict_gnn_lstm(gnn_model, X_gnn_train, A_static)  # (n_gnn_train, 24)
    gnn_test_preds  = predict_gnn_lstm(gnn_model, X_gnn_test,  A_static)  # (n_gnn_test, 24)

    # ── GNN evaluation ────────────────────────────────────────────────
    y_gnn_test_true = np.expm1(y_gnn_test)
    gnn_rows = []
    for h_idx, h in enumerate(HORIZONS):
        m = compute_metrics(y_gnn_test_true[:, h_idx], gnn_test_preds[:, h_idx])
        gnn_rows.append({'Model': 'C3_GNN_LSTM_pfxf', 'Station': station, 'Horizon': h, **m})
    gnn_df = pd.DataFrame(gnn_rows)
    gnn_df.to_csv(paths['results'] / 'C3_GNN_metrics.csv', index=False)

    # ── Stacking ensemble ─────────────────────────────────────────────
    stack_models = {}
    stack_rows = []
    base_model_classes = make_base_models()

    for h in tqdm(HORIZONS, desc=f'Stacking [{station}]'):
        X_tr_h, y_tr_h = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        h_idx = h - 1

        # Align GNN OOF predictions length with tabular training rows
        n_gnn = len(gnn_train_preds)
        n_tab = len(X_tr_h)
        if n_gnn <= n_tab:
            gnn_oof_h = np.full(n_tab, np.nan)
            gnn_oof_h[-n_gnn:] = gnn_train_preds[:, h_idx]
            valid = ~np.isnan(gnn_oof_h)
            X_h_valid = X_tr_h[valid]
            y_h_valid = y_tr_h[valid]
            gnn_oof_valid = gnn_oof_h[valid]
        else:
            gnn_oof_valid = gnn_train_preds[-n_tab:, h_idx]
            X_h_valid = X_tr_h
            y_h_valid = y_tr_h

        meta, _ = train_stacking_ensemble(
            X_h_valid, y_h_valid,
            gnn_oof_preds=gnn_oof_valid,
            save_path=paths['models'] / f'meta_learner_pfx_h{h}.pkl'
        )
        stack_models[h] = meta

    # ── Stacking test evaluation ──────────────────────────────────────
    meta_feat_cols = None  # determined per-horizon below

    for h in HORIZONS:
        X_tr_h, y_tr_h = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        X_te_h, y_te_h = fe.build_feature_matrix(test_df,  horizon=h, weather_mode=WEATHER_MODE)

        if meta_feat_cols is None:
            meta_feat_cols = [c for c in META_FEAT_COLS_BASE if c in X_te_h.columns]

        base_preds_test = []
        for name, factory in base_model_classes.items():
            m = factory()
            m.fit(X_tr_h.values, y_tr_h.values)
            base_preds_test.append(m.predict(X_te_h.values))

        h_idx = h - 1
        n_gnn_te = len(gnn_test_preds)
        n_tab_te = len(X_te_h)
        gnn_h = np.zeros(n_tab_te)
        if n_gnn_te >= n_tab_te:
            gnn_h = gnn_test_preds[-n_tab_te:, h_idx]
        else:
            gnn_h[-n_gnn_te:] = gnn_test_preds[:, h_idx]

        base_preds_test.append(gnn_h)
        base_arr  = np.column_stack(base_preds_test)
        meta_feats = X_te_h[meta_feat_cols].values if meta_feat_cols else np.zeros((n_tab_te, 1))

        final_preds = predict_stacking(stack_models[h], base_arr, meta_feats)
        m = compute_metrics(y_te_h.values, final_preds)
        stack_rows.append({'Model': 'C3_Stacking_pfxf', 'Station': station, 'Horizon': h, **m})

    stack_df = pd.DataFrame(stack_rows)
    stack_df.to_csv(paths['results'] / 'C3_Stacking_metrics.csv', index=False)

    # Combined B3 metrics (GNN + Stacking) — this file acts as the checkpoint
    combined = pd.concat([gnn_df, stack_df], ignore_index=True)
    combined.to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - wall_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore original TARGET and module state
cfg.TARGET = 'MzWarChrosci'
dl.TARGET = 'MzWarChrosci'
fe.TARGET = 'MzWarChrosci'
gs.TARGET = 'MzWarChrosci'
gs.ALL_STATIONS_ORDERED = ['MzWarChrosci'] + list(NEIGHBOR_STATIONS)

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')

11:19:03 | src.data_loader | INFO | Loading data from D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\data\raw\FINAL_merged_PM25_1g_all_seasons.csv



[1/7] MzWarChrosci — starting ...


11:19:04 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
11:19:04 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00
11:19:04 | src.data_loader | INFO | Train: 43824 rows (2019-01-01 00:00:00 → 2023-12-31 23:00:00)
11:19:04 | src.data_loader | INFO | Test : 8784 rows (2024-01-01 00:00:00 → 2024-12-31 23:00:00)
11:19:04 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -24 h
11:19:04 | src.models.gnn_stacking | INFO | build_gnn_sequence_dataset | seq_len=24 | weather_mode=perfect_forecast_full
11:19:04 | src.models.gnn_stacking | INFO | GNN sequences: X=(36467, 24, 7, 10) y=(36467, 24)
11:19:04 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 

  GNN Train: (36467, 24, 7, 10) | Test: (7495, 24, 7, 10)


11:19:10 | src.models.gnn_stacking | INFO | GNN-LSTM | seq_len=24  n_nodes=7  n_node_features=10  n_horizons=24  device=cpu
11:19:10 | src.models.gnn_stacking | INFO | Params: epochs=80  batch=64  patience=15  lr=0.00100
11:19:10 | src.models.gnn_stacking | INFO | Training samples=32821  val samples=3646
11:19:10 | src.models.gnn_stacking | INFO | -----------------------------------------------------------------
11:21:16 | src.models.gnn_stacking | INFO | Epoch   1/80  train=0.18613  val=0.06426  best=0.06426  lr=1.00e-03 *
11:22:54 | src.models.gnn_stacking | INFO | Epoch   2/80  train=0.08435  val=0.08319  best=0.06426  lr=1.00e-03  (no improve 1/15)
11:24:06 | src.models.gnn_stacking | INFO | Epoch   3/80  train=0.07620  val=0.06884  best=0.06426  lr=1.00e-03  (no improve 2/15)
11:25:13 | src.models.gnn_stacking | INFO | Epoch   4/80  train=0.07139  val=0.06798  best=0.06426  lr=1.00e-03  (no improve 3/15)
11:26:20 | src.models.gnn_stacking | INFO | Epoch   5/80  train=0.06767  val=

...

  GNN Train: (37102, 24, 6, 10) | Test: (7548, 24, 6, 10)


14:41:35 | src.models.gnn_stacking | INFO | Epoch   1/80  train=0.20428  val=0.05913  best=0.05913  lr=1.00e-03 *
14:42:18 | src.models.gnn_stacking | INFO | Epoch   2/80  train=0.09538  val=0.04941  best=0.04941  lr=1.00e-03 *
14:42:55 | src.models.gnn_stacking | INFO | Epoch   3/80  train=0.08813  val=0.07734  best=0.04941  lr=1.00e-03  (no improve 1/15)
14:43:32 | src.models.gnn_stacking | INFO | Epoch   4/80  train=0.08308  val=0.04739  best=0.04739  lr=1.00e-03 *
14:44:10 | src.models.gnn_stacking | INFO | Epoch   5/80  train=0.07734  val=0.05147  best=0.04739  lr=1.00e-03  (no improve 1/15)
14:44:51 | src.models.gnn_stacking | INFO | Epoch   6/80  train=0.07269  val=0.05373  best=0.04739  lr=1.00e-03  (no improve 2/15)
14:45:28 | src.models.gnn_stacking | INFO | Epoch   7/80  train=0.06845  val=0.05062  best=0.04739  lr=1.00e-03  (no improve 3/15)
14:46:09 | src.models.gnn_stacking | INFO | Epoch   8/80  train=0.06369  val=0.05086  best=0.04739  lr=1.00e-03  (no improve 4/15)
14:

Stacking [MzWarBajkowa]:   0%|          | 0/24 [00:00<?, ?it/s]

14:53:37 | src.feature_engineering | INFO | build_feature_matrix | horizon=1 | weather_mode=perfect_forecast_full
14:53:37 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -1 h
14:53:38 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
14:53:38 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
14:53:38 | src.feature_engineering | INFO |   X: (38496, 46) | y mean=18.12, y std=15.63
14:53:38 | src.models.gnn_stacking | INFO | OOF [1/4]: fitting base model 'xgb' across 5 folds
14:53:38 | src.models.gnn_stacking | INFO |   fold 1/5 — train=6187 val=6183
14:53:39 | src.models.gnn_stacking | INFO |   fold 2/5 — train=12370 val=6183
14:53:41 | src.models.gnn_stacking | INFO |   fold 3/5 — train=18553 val=6183
14:53:42 | src.models.gnn_stacking | INFO |   fold 4/5 — train=24736 val=6183
14:53:44 | src.models.gnn_stacking | INFO |   fold 5/5 — train=30919 v

[7/7] MzWarBajkowa — done | elapsed 26.7min | est. remaining 0.0min

All stations complete in 228.6min total.
